In [21]:
# STEP 1: Import libraries
from pathlib import Path
import pandas as pd

# STEP 2: Define your CSV file locations
data_path = Path(
    r"C:\Users\suraj\Downloads\henderson_clean_final_data.csv"
)

flags_path = Path(
    r"C:\Users\suraj\Downloads\henderson_measured_flags.csv"
)

# STEP 3: Check whether the files exist
for required in (data_path, flags_path):
    if not required.is_file():
        raise FileNotFoundError(
            f"Required file missing: {required}"
        )

# STEP 4: Load cleaned dataset
df = pd.read_csv(
    data_path,
    index_col="datetime",
    parse_dates=True
)

# STEP 5: Load measured-value flags
measured = pd.read_csv(
    flags_path,
    index_col="datetime",
    parse_dates=True
)

# STEP 6: Convert flags to Boolean values
for col in ["PM10", "NO2"]:

    if col not in df.columns or col not in measured.columns:
        raise KeyError(f"Missing column: {col}")

    if not pd.api.types.is_bool_dtype(measured[col]):
        converted = (
            measured[col]
            .astype("string")
            .str.strip()
            .str.lower()
            .map({
                "true": True,
                "false": False,
                "1": True,
                "0": False
            })
        )

        if converted.isna().any():
            raise ValueError(
                f"Invalid measurement flags in {col}"
            )

        measured[col] = converted.astype(bool)

# STEP 7: Prepare datetime indexes
df.index = pd.to_datetime(df.index)
measured.index = pd.to_datetime(measured.index)

# STEP 8: Check for duplicate timestamps
if df.index.has_duplicates or measured.index.has_duplicates:
    raise ValueError("Duplicate timestamps detected.")

# STEP 9: Sort by date and time
df = df.sort_index()
measured = measured.sort_index()

# STEP 10: Check dataset alignment
if not df.index.equals(measured.index):
    raise ValueError(
        "Cleaned data and measured flags do not have matching timestamps."
    )

# STEP 11: Display results
print("Data loaded successfully!")

print("\nDate range:")
print(df.index.min(), "to", df.index.max())

print("\nDataset shape:")
print(df.shape)

print("\nMeasured flags shape:")
print(measured.shape)

print("\nObserved pollutant shares:")
print(measured[["PM10", "NO2"]].mean().to_dict())

display(df.head())

Data loaded successfully!

Date range:
2006-01-01 00:00:00 to 2023-02-28 23:00:00

Dataset shape:
(150432, 5)

Measured flags shape:
(150432, 5)

Observed pollutant shares:
{'PM10': 0.9562659540523293, 'NO2': 0.8779913848117422}


,NO2,PM10,RH,Temp,WS
datetime,,,,,
2006-01-01 00:00:00,1.50,18.4,80.3,19.0,1.3
2006-01-01 01:00:00,0.76,32.6,81.1,19.0,3.1
2006-01-01 02:00:00,0.46,18.6,80.4,20.0,2.7
2006-01-01 03:00:00,0.42,15.1,79.4,20.0,1.6
2006-01-01 04:00:00,0.40,15.6,80.4,20.0,1.4


In [23]:

# FOUR SHORT-TERM AIR QUALITY FORECASTING MODELS
# NO2: Next 1 hour and next 24-hour average
# PM10: Next 1 hour and next 24-hour average

import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from lightgbm import LGBMRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# STEP 1: Prepare data
data = df[["PM10", "NO2"]].copy()
data.index = pd.to_datetime(data.index)
data = data.sort_index()

flags = measured[["PM10", "NO2"]].copy()
flags.index = pd.to_datetime(flags.index)
flags = flags.sort_index()

if data.index.has_duplicates or flags.index.has_duplicates:
    raise ValueError("Duplicate timestamps found.")

# Convert to regular hourly timestamps
hours = pd.date_range(data.index.min(), data.index.max(), freq="h")
data = data.reindex(hours)
flags = flags.reindex(hours)

for pollutant in ["PM10", "NO2"]:
    flags[pollutant] = (
        flags[pollutant]
        .astype("string")
        .str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
        .fillna(False)
        .astype(bool)
    )

# STEP 2: Create features from historical data
def create_features(target):
    other = "NO2" if target == "PM10" else "PM10"

    X = pd.DataFrame(index=data.index)

    X["hour"] = data.index.hour
    X["dayofweek"] = data.index.dayofweek
    X["month"] = data.index.month
    X["dayofyear"] = data.index.dayofyear

    # Lag features: use current and previous observations
    for lag in [0, 1, 2, 3, 24]:
        X[f"{target}_lag{lag}"] = data[target].shift(lag)

    # Rolling historical features
    X[f"{target}_roll3"] = (
        data[target].rolling(3, min_periods=3).mean()
    )
    X[f"{target}_roll24"] = (
        data[target].rolling(24, min_periods=24).mean()
    )
    X[f"{target}_std24"] = (
        data[target].rolling(24, min_periods=24).std()
    )

    # Other pollutant
    X[f"{other}_lag0"] = data[other]
    X[f"{other}_lag1"] = data[other].shift(1)

    return X

# STEP 3: Define forecast targets
forecasts = [
    ("NO2", 1),
    ("PM10", 1),
    ("NO2", 24),
    ("PM10", 24)
]

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

results = []

# STEP 4: Train four models
for target, horizon in forecasts:

    print(f"\nTraining {target}: next {horizon} hour(s)")

    X = create_features(target)

    # Future target: next hour OR mean of next 24 hours
    y = (
        data[target]
        .rolling(horizon, min_periods=horizon)
        .mean()
        .shift(-horizon)
    )

    # Require genuinely observed future targets
    future_observed = (
        flags[target]
        .astype(int)
        .rolling(horizon, min_periods=horizon)
        .sum()
        .shift(-horizon)
        .eq(horizon)
    )

    # Require genuinely observed input readings
    valid_inputs = (
        flags[target]
        & flags[target].shift(1).fillna(False)
        & flags[target].shift(2).fillna(False)
        & flags[target].shift(3).fillna(False)
        & flags[target].shift(24).fillna(False)
        & flags[target].rolling(24, min_periods=24).sum().eq(24)
    )

    valid = valid_inputs & future_observed & X.notna().all(axis=1) & y.notna()

    X = X.loc[valid]
    y = y.loc[valid]

    # STEP 5: Chronological data splitting
    n = len(X)
    train_end = int(n * 0.60)
    val_end = int(n * 0.80)

    # Gap avoids target overlap across splits
    gap = horizon

    X_train = X.iloc[:train_end-gap]
    y_train = y.iloc[:train_end-gap]

    X_val = X.iloc[train_end:val_end-gap]
    y_val = y.iloc[train_end:val_end-gap]

    X_test = X.iloc[val_end:]
    y_test = y.iloc[val_end:]

    if min(len(X_train), len(X_val), len(X_test)) < 50:
        raise ValueError(f"Insufficient data for {target} {horizon}h")

    # STEP 6: Train LightGBM
    model = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    # Validation predictions
    val_pred = np.maximum(model.predict(X_val), 0)

    val_rmse = np.sqrt(
        mean_squared_error(y_val, val_pred)
    )

    # Retrain using training + validation data
    X_train_final = X.iloc[:val_end-gap]
    y_train_final = y.iloc[:val_end-gap]

    final_model = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    )

    final_model.fit(X_train_final, y_train_final)

    # STEP 7: Test evaluation
    predictions = np.maximum(final_model.predict(X_test), 0)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    # STEP 8: Save the model
    filename = model_dir / f"forward_{target}_{horizon}h.pkl"

    joblib.dump({
        "model": final_model,
        "features": list(X.columns),
        "target": target,
        "horizon_hours": horizon
    }, filename)

    results.append({
        "Forecast": f"{target} next {horizon}h",
        "Validation RMSE": round(val_rmse, 3),
        "Test MAE": round(mae, 3),
        "Test RMSE": round(rmse, 3),
        "Test R2": round(r2, 3)
    })

    print("Saved:", filename)

# STEP 9: Display results
results_df = pd.DataFrame(results)

print("\nFINAL MODEL RESULTS")
display(results_df)

print("\nAll four models trained and saved successfully!")
print("Model folder:", model_dir.resolve())



Training NO2: next 1 hour(s)
Saved: models\forward_NO2_1h.pkl

Training PM10: next 1 hour(s)
Saved: models\forward_PM10_1h.pkl

Training NO2: next 24 hour(s)
Saved: models\forward_NO2_24h.pkl

Training PM10: next 24 hour(s)
Saved: models\forward_PM10_24h.pkl

FINAL MODEL RESULTS


,Forecast,Validation RMSE,Test MAE,Test RMSE,Test R2
0,NO2 next 1h,4.847,2.531,4.121,0.762
1,PM10 next 1h,5.185,4.361,6.192,0.343
2,NO2 next 24h,6.157,5.197,6.965,0.254
3,PM10 next 24h,3.451,2.733,3.564,0.283



All four models trained and saved successfully!
Model folder: C:\Users\suraj\models
